# Setup

In [ ]:

%run notebook_setup.py
import pandas as pd
import numpy as np
from scipy.stats import trim_mean
import polars as pl

from src.config.dir_config import OUTPUT_PATH_DEMMAND_SUMMARY,OUTPUT_PATH_PROCESSED_WEEKLY_SALES
from src.config.bigquery_config import CREDENTIALS_GBQ, PROJECT_ID_GBQ
from src.utils.read_data import read_data
from src.utils.setup_logging import setup_logging
from src.analysis.demand_summary import demand_classification
from src.models.croston import croston_calculate
from src.models.croston_tsb import calculate_tsb_forecast

import logging

setup_logging()

# Data import

In [ ]:
genex = pd.read_parquet('/bi/workspace/Projects/Forecast/forecast/data/processed/data_genex.parquet')

In [ ]:
#weekly_sales_df = pd.read_parquet('../data/processed/weekly_sales_processed.parquet')

In [ ]:
demand_summary = pd.read_parquet(OUTPUT_PATH_DEMMAND_SUMMARY)

In [ ]:
demand_summary.head(2)

# Data analysis

In [ ]:
genex_summary = genex.groupby(['nombre_temporada','ano_temporada','clasif','factor','semana_vta','inicio_semana'],
              observed=True).agg(
    casos=('nombre_sucursal', 'count'),
    vta_promedio=('vta_promedio', 'mean'),
    can_original=('can_original', 'mean')).round(2).reset_index().to_excel('../sandbox/genex_summary.xlsx', index=False)

In [ ]:
factor = 1
sv = 8
nombre_temporada = 'Invierno'
clasif = 'BOTTOM INTELIGENTE'
inicio_semana = '2025-04-14'
#cod_producto = 272799

genex_ex= genex.query(f'nombre_temporada == "{nombre_temporada}" and '
                            f'clasif == "{clasif}" and '
                            f'factor == {factor} and '
                            f'semana_vta == {sv} and '
                            f'inicio_semana == "{inicio_semana}" and '
                            f'can_final > 0').reset_index(drop=True)



genex_ex = genex_ex.dropna(subset=['cod_talla'])

Index interesantes

-   223
-   3672
-   3942 (caso con venta menos de 7 dias)

In [ ]:
# Calcular columna de venta estimada
genex_ex = genex_ex.copy()  # buena práctica al modificar columnas
genex_ex['vta_estimada'] = (
    genex_ex['vta_promedio'] * genex_ex['factor'] * genex_ex['semana_vta']
).round(0)

# Columnas que nos interesan
interest_cols = [
    'nombre_depto','nombre_linea','nombre_sucursal', 'cod_producto', 'cod_talla', 'cod_sucursal', 'inicio_semana',
    'vta_periodo', 'vta_promedio', 'factor', 'semana_vta', 'vta_estimada',
    'stock_sucursal', 'repo_x_dda', 'can_final'
]

# Tomar una fila aleatoria
random_index = genex_ex.sample(1).index[0]
genex_row = genex_ex.loc[random_index, interest_cols].to_dict()

# Agregar índice original
genex_row['index'] = int(random_index)

# Guardar como filtro
filtro = genex_row

# Mostrar
filtro

| Parámetro | Valor sugerido | Justificación                                                                   |
| --------- | -------------- | ------------------------------------------------------------------------------- |
| **alpha** | **0.2**        | Reacciona moderadamente a cambios en la **cantidad** de unidades vendidas.      |
| **beta**  | **0.1**        | Mantiene estable la estimación de la **probabilidad de venta** semana a semana. |


In [ ]:
weekly_sales_sample = weekly_sales_df[
    (weekly_sales_df['cod_producto'] == filtro['cod_producto']) &
    (weekly_sales_df['cod_talla'] == filtro['cod_talla']) &
    (weekly_sales_df['cod_sucursal'] == filtro['cod_sucursal'])
].copy()

weekly_sales_sample['date'] = pd.to_datetime(
    weekly_sales_sample['cod_ano_comercial'].astype(str) + '-' +
    weekly_sales_sample['cod_semana'].astype(str) + '-1',  # 1 is the first day of the week
    format='%Y-%W-%w'
).dt.date


weekly_sales_sample['tricot_estimate'] = (
    weekly_sales_sample['weekly_sales']
    .shift(1)  # desplaza una semana hacia atrás para excluir la semana actual
    .rolling(window=4, min_periods=4)
    .mean()
).fillna(0).round(0).astype(int)


weekly_sales_sample = croston_calculate(weekly_sales_sample)

weekly_sales_sample = calculate_tsb_forecast(weekly_sales_sample, alpha=0.2, beta=0.5)



weekly_sales_sample = weekly_sales_sample[['date','nombre_sucursal','nombre_depto','nombre_linea','cod_producto','nom_talla','weekly_available_stock','flag_inventory_available','weekly_sales','croston_estimate','tsb_forecast','tricot_estimate']].round(2)

weekly_sales_sample

In [ ]:
cutoff_date = pd.to_datetime('2025-04-28')
cobertura = 8

# Convertir a datetime si no lo está
weekly_sales_sample['date'] = pd.to_datetime(weekly_sales_sample['date'])

# Filtrar las siguientes 8 semanas (no incluye la semana de cutoff si quieres post-ventas)
future_data = weekly_sales_sample[
    (weekly_sales_sample['date'] > cutoff_date) &
    (weekly_sales_sample['date'] <= cutoff_date + pd.Timedelta(weeks=cobertura))
]

# Tomar el forecast de la semana de corte y multiplicarlo por 8
forecast_base = weekly_sales_sample[weekly_sales_sample['date'] == cutoff_date]

if forecast_base.empty or future_data.empty:
    print("⚠️ No hay datos para la fecha de corte o las semanas siguientes.")
else:
    forecast_8w = (forecast_base[['tricot_estimate', 'tsb_forecast', 'croston_estimate']] * cobertura).iloc[0].round(1)

    # Sumar las ventas reales de las 8 semanas siguientes
    real_value = future_data['weekly_sales'].sum()

    weeks_with_inventory = future_data['flag_inventory_available'].sum()

    # Combinar en un diccionario
    forecast_results = {
        'Semanas con inventario': weeks_with_inventory,
        
        f'Ventas reales {cobertura} semanas': int(real_value),
        'Forecast genex' : int(filtro['vta_estimada']),
        'Forecast Tricot (MA)': int(forecast_8w['tricot_estimate']),
        
        'Forecast TSB': int(forecast_8w['tsb_forecast']),
        'Forecast Croston': int(forecast_8w['croston_estimate'])
    }

    # Mostrar resultados ordenados
    for k, v in forecast_results.items():
        print(f"{k}: {v}")